In [8]:
# import all necessary libraries

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
from torch import nn
import torchvision
import numpy as np

import matplotlib.pyplot as plt
print(torch.__version__)

1.10.0


In [9]:
path = "/home/mingdayang/mmdetection3d/outputs/train/unibev_nus_LC_cnw_256_modality_dropout_freeze_unibev_train_auxiliary_bev_mapping/latest.pth"

In [10]:
checkpoint = torch.load(path, map_location='cpu')
print(checkpoint.keys())
print(checkpoint['state_dict'].keys())

dict_keys(['meta', 'state_dict', 'optimizer'])
odict_keys(['pts_middle_encoder.conv_input.0.weight', 'pts_middle_encoder.conv_input.1.weight', 'pts_middle_encoder.conv_input.1.bias', 'pts_middle_encoder.conv_input.1.running_mean', 'pts_middle_encoder.conv_input.1.running_var', 'pts_middle_encoder.conv_input.1.num_batches_tracked', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.conv1.weight', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn1.weight', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn1.bias', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn1.running_mean', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn1.running_var', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn1.num_batches_tracked', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.conv2.weight', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn2.weight', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn2.bias', 'pts_middle_encoder.encoder_layers.encoder_layer1.0.bn2.ru

In [11]:
# Search for bev_consumer parameters
bev_consumer_keys = [k for k in checkpoint['state_dict'].keys() if 'bev_consumer' in k]
print(f"Found {len(bev_consumer_keys)} bev_consumer parameters:")
for key in bev_consumer_keys:
    print(f"  {key}: shape={checkpoint['state_dict'][key].shape}")

Found 4 bev_consumer parameters:
  pts_bbox_head.bev_consumer.mlp.0.weight: shape=torch.Size([256, 256])
  pts_bbox_head.bev_consumer.mlp.0.bias: shape=torch.Size([256])
  pts_bbox_head.bev_consumer.mlp.2.weight: shape=torch.Size([256, 256])
  pts_bbox_head.bev_consumer.mlp.2.bias: shape=torch.Size([256])


## Extract BEV Consumer Parameters for Inference

To use the trained MyBEVConsumer for standalone inference, we need to:
1. Define the model architecture
2. Extract and rename the parameters (remove prefix)
3. Load into the model
4. Run inference

In [12]:
# Step 1: Define the MyBEVConsumer architecture (must match training)
class MyBEVConsumer(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )
    
    def forward(self, bev_embed):
        """
        Args:
            bev_embed: (bs, bev_h*bev_w, embed_dim) camera BEV features
        Returns:
            predicted lidar BEV features with same shape
        """
        return self.mlp(bev_embed)

# Instantiate the model
model = MyBEVConsumer(embed_dim=256)
print(f"Model architecture:\n{model}")

Model architecture:
MyBEVConsumer(
  (mlp): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
)


In [13]:
# Step 2: Extract bev_consumer parameters and remove prefix
bev_consumer_state_dict = {}
prefix = "pts_bbox_head.bev_consumer."

for key in checkpoint['state_dict'].keys():
    if key.startswith(prefix):
        # Remove the prefix to get the model's parameter name
        new_key = key[len(prefix):]
        bev_consumer_state_dict[new_key] = checkpoint['state_dict'][key]

print(f"Extracted {len(bev_consumer_state_dict)} parameters:")
for key, value in bev_consumer_state_dict.items():
    print(f"  {key}: {value.shape}")

Extracted 4 parameters:
  mlp.0.weight: torch.Size([256, 256])
  mlp.0.bias: torch.Size([256])
  mlp.2.weight: torch.Size([256, 256])
  mlp.2.bias: torch.Size([256])


In [14]:
# Step 3: Load the parameters into the model
model.load_state_dict(bev_consumer_state_dict)
model.eval()  # Set to evaluation mode
print("Parameters loaded successfully!")
print(f"Model is in eval mode: {not model.training}")

Parameters loaded successfully!
Model is in eval mode: True


In [ ]:
# Step 4: Test inference with dummy data
# Create dummy camera BEV features (batch_size=2, bev_h=200, bev_w=200, embed_dim=256)
batch_size = 2
bev_h, bev_w = 200, 200
embed_dim = 256

dummy_camera_bev = torch.randn(batch_size, bev_h * bev_w, embed_dim)

# Run inference
with torch.no_grad():
    predicted_lidar_bev = model(dummy_camera_bev)

print(f"Input shape: {dummy_camera_bev.shape}")
print(f"Output shape: {predicted_lidar_bev.shape}")
print(f"Output mean: {predicted_lidar_bev.mean().item():.4f}, std: {predicted_lidar_bev.std().item():.4f}")